# Experiment 2: Octic ViT ($D_4$-Equivariant Encoder)

## Symmetry Group

The **dihedral group** $D_4 = \langle r, s \mid r^4 = s^2 = e,\; srs = r^{-1} \rangle$ of order 8. Acts by rotating and reflecting the spatial axes by multiples of $90^\circ$.

## Architecture (Nordstr\"om et al., 2025)

Each ViT layer is replaced with a $D_4$-equivariant variant. By **Schur's lemma**, equivariant linear maps act independently on each irreducible component:

$$\mathbb{R}^d \;\cong\; \underbrace{(\mathbb{R}^{d/8})_{A_1} \oplus (\mathbb{R}^{d/8})_{A_2} \oplus (\mathbb{R}^{d/8})_{B_1} \oplus (\mathbb{R}^{d/8})_{B_2}}_{\text{four 1-dim irreps}} \oplus \underbrace{(\mathbb{R}^{d/4})_E}_{\text{one 2-dim irrep}}$$

Invariant pooling projects onto the trivial component $A_1$, yielding a $D_4$-invariant output:

$$\varphi_\theta(g \cdot x) = \varphi_\theta(x) \quad \forall g \in D_4$$

## Siamese Pipeline

$$f_\theta(x_1, x_2) = \sigma\!\bigl(h_\psi(\varphi_\theta(x_1),\, \varphi_\theta(x_2))\bigr)$$

## Loss

$$\mathcal{L} = \mathcal{L}_{\mathrm{BCE}} + \lambda\,\mathcal{L}_{\mathrm{contr}}^{L^2}, \quad \mathcal{L}_{\mathrm{contr}}^{L^2}(z_1, z_2, y) = y\,\|z_1 - z_2\|_2^2 + (1 - y)\,\max(0, m - \|z_1 - z_2\|_2)^2$$

## Runs

| Run | Augmentation | $\lambda$ |
|-----|-------------|----------|
| `octic_vit_no_reg` | Block P only | 0.0 |
| `octic_vit_reg` | Block P only | 0.1 |

**Key hypothesis**: $D_4$-invariant encoder trained *without* Block G should match or beat ViT-L/16 *with* Block G on R90/R180/R270 recall.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import yaml
import logging
import matplotlib.pyplot as plt

from src.utils.seed import set_seed
from src.data_module.augmentations import build_train_transform, build_val_transform
from src.data_module.coco_dataset import build_coco_dataloaders
from src.data_module.domainnet_dataset import DomainNetEvalDataset
from src.encoders import EncoderFactory
from src.losses.composite import CompositeLoss
from src.models.siamese import SiameseNet
from src.training.trainer import Trainer
from src.validation.evaluator import DomainNetEvaluator
from src.validation.tsne import compute_tsne_embeddings, plot_tsne
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

set_seed(cfg['seed'])
train_tfm = build_train_transform(cfg)
val_tfm = build_val_transform(cfg)
dataloaders = build_coco_dataloaders(cfg, train_tfm, val_tfm)

In [ ]:
# --- OCTIC_WEIGHTS: set path to your D4 DINOv2 checkpoint ---
OCTIC_WEIGHTS = None  # e.g. '/data/weights/d8_inv_early_dinov2_large_patch16.pth'

histories = {}
models = {}

for run_name, lam in [('octic_vit_no_reg', 0.0), ('octic_vit_reg', 0.1)]:
    print(f'\n{"="*60}')
    print(f'Run: {run_name}, lambda={lam}')
    print(f'{"="*60}')
    
    set_seed(cfg['seed'])
    encoder = EncoderFactory('octic_vit', freeze=True, weights_path=OCTIC_WEIGHTS)
    model = SiameseNet(encoder, hidden_dim=512, dropout=0.1)
    
    criterion = CompositeLoss(
        bce_weight_pos=0.3, bce_weight_neg=0.7,
        contrastive_margin=1.0, lambda_reg=lam,
    )
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)
    
    trainer = Trainer(
        model=model, criterion=criterion, optimizer=optimizer,
        scheduler=scheduler, device=device,
        checkpoint_dir=f'../outputs/checkpoints/{run_name}',
    )
    histories[run_name] = trainer.fit(
        dataloaders['train'], dataloaders['val'],
        num_epochs=cfg['training']['num_epochs'],
        run_name=run_name,
    )
    models[run_name] = model

## DomainNet Evaluation

In [ ]:
domainnet_ds = DomainNetEvalDataset(
    root=cfg['data']['domainnet_root'],
    domains=cfg['data']['domainnet_domains'],
    pairs_per_domain=cfg['data']['pairs_per_domain'],
)

for run_name, model in models.items():
    print(f'\n--- {run_name} ---')
    evaluator = DomainNetEvaluator(model=model, dataset=domainnet_ds, device=device)
    report = evaluator.run()
    print(json.dumps(report, indent=2))

## t-SNE: Octic ViT Embeddings

In [ ]:
last_model = list(models.values())[-1]
tsne_loader = DataLoader(domainnet_ds, batch_size=32, shuffle=False, num_workers=2)
coords, domains = compute_tsne_embeddings(last_model, tsne_loader, device=device)
fig = plot_tsne(coords, domains, title='t-SNE: Octic ViT ($D_4$)',
                save_path='../outputs/figures/octic_vit_tsne.pdf')
plt.show()

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    epochs = [h['epoch'] for h in hist]
    axes[0].plot(epochs, [h['train']['loss'] for h in hist], label=f'{name} (train)')
    axes[1].plot(epochs, [h['val']['loss'] for h in hist], label=f'{name} (val)')
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')
fig.tight_layout()
fig.savefig('../outputs/figures/octic_vit_curves.pdf', dpi=300)
plt.show()